# Lab 19-13 · Testbenches, Coverage, BDDs and Model Checking

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import verif

## Part 1 · A constrained-random testbench in Verilog

*Predict* the worst wait (in grants to others) for the round-robin arbiter, and whether the fixed-priority copy breaks the bound of 3.

In [2]:
if shutil.which("iverilog"):
    import os, tempfile
    sim = os.path.join(tempfile.mkdtemp(), "rr_sim.vvp")
    subprocess.run(["iverilog", "-g2012", "-o", sim, "rr_arbiter_tb.v", "rr_arbiter.v"], check=True)
    print(subprocess.run(["vvp", sim], capture_output=True, text=True).stdout)
else:
    print("Icarus Verilog not installed; the README lists the expected output")
print("exhaustive worst wait, round-robin, N = 2..5:", [verif.arbiter_worst_wait(n) for n in range(2, 6)])
print("fixed priority, N = 4 (capped at 12 = unbounded):", verif.arbiter_worst_wait(4, fixed=True))

coverage: 9 of 9 bins; grants 501 509 491 495; active-request counts 4 77 428 907 584
round-robin: 0 errors, 0 fairness violations, worst wait 3 grants
fixed-priority variant: 2733 fairness violations
TEST PASSED
rr_arbiter_tb.v:79: $finish called at 20016000 (1ps)

exhaustive worst wait, round-robin, N = 2..5: [1, 2, 3, 4]
fixed priority, N = 4 (capped at 12 = unbounded): 12


## Part 2 · Coverage closure

*Predict* the mean samples to fill 16 equal bins, and 8 bins with one bin of probability 0.01.

In [3]:
equal = [1 / 16] * 16
rare = [0.99 / 7] * 7 + [0.01]
for name, p in (("16 equal bins", equal), ("rare bin", rare)):
    print(f"{name}: exact {verif.expected_to_cover_exact(p):.2f}, simulated {verif.simulate_cover(p, trials=2000):.2f}")
t = np.arange(0, 400)
for p, label in ((equal, "16 equal bins"), (rare, "7 common + 1 rare")):
    plt.plot(t, [sum(1 - (1 - q) ** k for q in p) / len(p) for k in t], label=label)
plt.xlabel("random samples"); plt.ylabel("expected fraction of bins covered"); plt.legend(); plt.show()

16 equal bins: exact 54.09, simulated 54.61


rare bin: exact 101.89, simulated 102.22


## Part 3 · BDDs and variable order

*Predict* the node counts of $\\sum x_i y_i$ and of the equality comparator under the two orders.

In [4]:
def build(n, kind, order):
    b = verif.BDD(2 * n)
    if order == "interleaved":
        xs, ys = [b.var(2 * i) for i in range(n)], [b.var(2 * i + 1) for i in range(n)]
    else:
        xs, ys = [b.var(i) for i in range(n)], [b.var(n + i) for i in range(n)]
    f = 0 if kind == "sum" else 1
    for x, y in zip(xs, ys):
        f = b.OR(f, b.AND(x, y)) if kind == "sum" else b.AND(f, b.NOT(b.XOR(x, y)))
    return b.size(f)

for kind in ("sum", "eq"):
    print(kind, [(n, build(n, kind, "interleaved"), build(n, kind, "separated")) for n in range(3, 9)])
b = verif.BDD(16)
A, B = [b.var(2 * i) for i in range(8)], [b.var(2 * i + 1) for i in range(8)]
print("8-bit ripple adder, nodes per output bit:", [b.size(o) for o in verif.ripple_adder(b, A, B)])

sum [(3, 6, 14), (4, 8, 30), (5, 10, 62), (6, 12, 126), (7, 14, 254), (8, 16, 510)]
eq [(3, 9, 21), (4, 12, 45), (5, 15, 93), (6, 18, 189), (7, 21, 381), (8, 24, 765)]
8-bit ripple adder, nodes per output bit: [3, 6, 9, 12, 15, 18, 21, 24, 23]


## Part 4 · Equivalence checking with a miter

*Predict* the counterexample for the lookahead adder with the $p_2 p_1 g_0$ term dropped from $c_3$.

In [5]:
b = verif.BDD(8)
A, B = [b.var(2 * i) for i in range(4)], [b.var(2 * i + 1) for i in range(4)]
print("correct adders equivalent:", verif.equivalent(b, verif.ripple_adder(b, A, B), verif.lookahead_adder(b, A, B))[0])
eq, cex = verif.equivalent(b, verif.ripple_adder(b, A, B), verif.lookahead_adder(b, A, B, bug_bit=3))
a = sum(cex.get(2 * i, 0) << i for i in range(4)); bb = sum(cex.get(2 * i + 1, 0) << i for i in range(4))
print("buggy adder equivalent:", eq, "; counterexample A =", a, "B =", bb, "; A + B =", a + bb)

correct adders equivalent: True
buggy adder equivalent: False ; counterexample A = 7 B = 1 ; A + B = 8


## Part 5 · Model checking

*Predict* the reachable states, the BMC depth of the FIFO bug, the smallest $k$ for $x \\ne 7$, and the fate of the two-process protocol.

In [6]:
for buggy in (False, True):
    init, succ = verif.fifo_counter(buggy=buggy)
    holds, trace = verif.check_invariant(init, succ, lambda c: c <= 4)
    print(f"FIFO buggy={buggy}: reachable {sorted(verif.reachable(init, succ))}, holds {holds}, trace {trace}, "
          f"BMC depth {verif.bmc(init, succ, lambda c: c <= 4, 10)}")
init, succ = verif.step_by_two()
print("add-by-two reachable:", sorted(verif.reachable(init, succ)))
print("x != 7 k-inductive for k = 1..5:", [verif.k_inductive(range(8), succ, lambda x: x != 7, k) for k in range(1, 6)])
for buggy in (False, True):
    init, succ, states = verif.mutex(buggy)
    holds, trace = verif.check_invariant(init, succ, lambda s: not (s[0] == "C" and s[1] == "C"))
    print(f"mutex buggy={buggy}: {len(verif.reachable(init, succ))} of {len(states)} states reachable, "
          f"mutual exclusion {holds}, counterexample {trace}")

FIFO buggy=False: reachable [0, 1, 2, 3, 4], holds True, trace None, BMC depth None
FIFO buggy=True: reachable [0, 1, 2, 3, 4, 5, 6, 7], holds False, trace [0, 1, 2, 3, 4, 5], BMC depth 5
add-by-two reachable: [0, 2, 4, 6]
x != 7 k-inductive for k = 1..5: [False, False, False, True, True]
mutex buggy=False: 16 of 18 states reachable, mutual exclusion True, counterexample None
mutex buggy=True: 18 of 18 states reachable, mutual exclusion False, counterexample [('I', 'I', 0), ('R', 'R', 0), ('C', 'C', 0)]


## Discussion

The questions and worked answers are in the [handout](README.md#discussion).